# Aula 17 - Criando Agentes de IA Locais com Foundry Local e Qwen

Neste notebook você constrói um **assistente de engenharia local** que roda totalmente na sua estação de trabalho. Nenhuma inferência na nuvem é usada em nenhum momento. O assistente pode:

1. **Chamar ferramentas** via chamada de função Qwen através do Foundry Local.
2. **Listar e ler arquivos** dentro de um diretório de projeto isolado.
3. **Analisar código** com métricas simples.
4. **Pesquisar documentação** com RAG local (Chroma).
5. **Usar um servidor MCP local** (ignorado graciosamente se nenhum estiver configurado).

O código do agente é quase idêntico às aulas na nuvem — apenas o endpoint do cliente muda da nuvem para `localhost`.


## Configuração

Antes de executar este notebook:

1. **Instale o Microsoft Foundry Local** (consulte a [documentação](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) para seu sistema operacional).
2. **Baixe e inicie um modelo Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Instale os pacotes Python abaixo.

Tudo roda localmente. Uma máquina com ~8 GB de RAM é um mínimo realista.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Conecte-se ao Foundry Local

`FoundryLocalManager` baixa o modelo se necessário, inicia o serviço local e nos fornece um **endpoint compatível com OpenAI**. Em seguida, direcionamos o SDK padrão da OpenAI para ele. A chave da API é um marcador local — nenhuma credencial de nuvem está envolvida.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Ferramentas Locais (Operações de Arquivo em Sandbox)

Criamos um pequeno projeto de exemplo no disco, e então definimos ferramentas limitadas a essa raiz do projeto. A verificação de sandbox é importante mesmo localmente: uma ferramenta que lê caminhos arbitrários é executada com as permissões do seu usuário e pode acessar qualquer coisa que você consiga.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG Local com Chroma

Incorporamos um pequeno conjunto de trechos de documentação em uma coleção local do Chroma. O Chroma roda no processo e armazena vetores no disco — sem servidor, sem nuvem. A ferramenta `search_docs` recupera os trechos mais relevantes para uma consulta.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. O Loop de Chamada de Ferramentas

Agora registramos as ferramentas com o modelo usando o esquema de ferramentas OpenAI e executamos o loop padrão de chamada de ferramentas — o modelo solicita uma ferramenta, nós a executamos localmente, fornecemos o resultado de volta e repetimos até que o modelo produza uma resposta final. A chamada de função confiável do Qwen é o que faz isso funcionar no dispositivo.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP Local (Opcional)

MCP é um transporte, não um serviço em nuvem — um servidor MCP pode ser executado como um processo local via `stdio`. A célula abaixo mostra como você se conectaria a um servidor MCP local se você tiver um configurado (por exemplo, um servidor de sistema de arquivos). Ela pula de forma graciosa quando `LOCAL_MCP_COMMAND` não está definido, então o notebook ainda roda completamente sem ele.

Nota de segurança: um servidor MCP local roda com as permissões do seu usuário. Restringa-o a um diretório de projeto e valide suas saídas antes de agir sobre elas.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Resumo

Você construiu um assistente de engenharia que roda inteiramente em sua máquina:

- **Foundry Local** serviu um modelo **Qwen** por trás de um endpoint compatível com OpenAI — assim o código do agente corresponde às lições da nuvem.
- **Ferramentas em sandbox** deram ao agente acesso a arquivos e análise de código sem sair do diretório do projeto.
- **Chroma** forneceu **RAG local** sobre a documentação.
- **MCP Local** mostrou como reutilizar o ecossistema MCP offline.

Nenhum uso de inferência na nuvem foi feito em momento algum.

### Desafio

Estenda o agente local para:

1. **Trabalhar com múltiplos servidores MCP** — conectar um servidor de sistema de arquivos e um servidor git e deixar o agente escolher entre eles.
2. **Usar memória local** — persistir um histórico curto de conversas no disco para que o assistente lembre de turnos anteriores mesmo após reinícios do notebook.
3. **Suportar orquestração multi-agente local** — adicionar um segundo agente local (por exemplo, um revisor) e fazer com que os dois colaborem em uma tarefa.

Na próxima lição você aprenderá como proteger agentes de IA implantados.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
